# Phase 5: bounded CM population survey

This notebook analyzes all 4,165 candidates in the predefined binary and ternary Hermitian bounds. It asks whether distance and passive logical symmetry are associated *within this bounded CM population*. This is not a canonical average over the CM locus and not yet a certified CM-versus-non-CM population comparison.

In [1]:
import sys
from pathlib import Path

here = Path.cwd().resolve()
candidates = (here, here.parent, here / 'passive-cliffords')
project = next(path for path in candidates if (path / 'src' / 'gkp_passive_cliffords').exists())
workspace = project.parent
sys.path.insert(0, str(project / 'src'))
sys.path.insert(0, str(workspace / 'src'))

from gkp_passive_cliffords import load_phase5_population_ledger
rows, summaries = load_phase5_population_ledger(project / 'data')
print(f'Loaded {len(rows):,} exact CM candidate records.')

Loaded 4,165 exact CM candidate records.


## Aggregate result

The generic baseline is the image of $\{\pm I\}$ on $K(L)$. `Enhanced` means a strictly larger passive logical image.

In [2]:
columns = [
    'polarization_type', 'candidate_count',
    'extra_passive_symmetry_count', 'extra_passive_symmetry_fraction',
    'upper_quartile_extra_symmetry_fraction',
    'mean_ell_squared_enhanced', 'mean_ell_squared_minimal_image',
    'distance_log_image_correlation',
]
print(' | '.join(columns))
print(' | '.join(['---'] * len(columns)))
for row in summaries:
    values = []
    for column in columns:
        value = row[column]
        values.append(f'{value:.6f}' if isinstance(value, float) else str(value))
    print(' | '.join(values))

polarization_type | candidate_count | extra_passive_symmetry_count | extra_passive_symmetry_fraction | upper_quartile_extra_symmetry_fraction | mean_ell_squared_enhanced | mean_ell_squared_minimal_image | distance_log_image_correlation
--- | --- | --- | --- | --- | --- | --- | ---
[1, 1, 2] | 1051 | 131 | 0.124643 | 0.228137 | 0.656200 | 0.458680 | 0.326950
[1, 1, 3] | 1070 | 106 | 0.099065 | 0.227612 | 0.586101 | 0.361024 | 0.442292
[1, 2, 2] | 253 | 203 | 0.802372 | 0.906250 | 0.401910 | 0.301090 | 0.663158
[1, 3] | 876 | 43 | 0.049087 | 0.086758 | 0.534008 | 0.354682 | 0.221766
[1, 5] | 915 | 38 | 0.041530 | 0.113537 | 0.385885 | 0.249532 | 0.199360


## Logical image distributions

In [3]:
for row in summaries:
    print(tuple(row['polarization_type']), row['logical_image_histogram'])

(1, 1, 2) {'1': 920, '2': 84, '3': 37, '6': 10}
(1, 1, 3) {'2': 964, '4': 37, '6': 61, '24': 8}
(1, 2, 2) {'1': 50, '2': 144, '3': 1, '4': 4, '6': 29, '8': 10, '12': 3, '18': 10, '48': 2}
(1, 3) {'2': 833, '4': 16, '6': 26, '24': 1}
(1, 5) {'2': 877, '4': 25, '6': 12, '8': 1}


## Distance--gate Pareto candidates

A candidate is Pareto optimal if no retained candidate of the same type has both at least as large $\ell^2$ and at least as large a passive image, with one strict improvement. Remaining duplicate representatives reflect the deliberately partial ternary isometry reduction.

In [4]:
for summary in summaries:
    D = summary['polarization_type']
    pareto = [row for row in rows if row['polarization_type'] == D and row['pareto_optimal']]
    signatures = sorted({(row['ell_squared_exact'], row['logical_image_order']) for row in pareto})
    print(tuple(D), 'Pareto signatures:', signatures)

(1, 1, 2) Pareto signatures: [('2/sqrt(4)', 6)]
(1, 1, 3) Pareto signatures: [('2/sqrt(3)', 24)]
(1, 2, 2) Pareto signatures: [('2/sqrt(4)', 48)]
(1, 3) Pareto signatures: [('4/sqrt(24)', 24)]
(1, 5) Pareto signatures: [('16/5/sqrt(43)', 6), ('2/sqrt(20)', 8), ('4/sqrt(55)', 4)]


## Reproducibility checks

In [5]:
expected_counts = {(1,3): 876, (1,5): 915, (1,1,2): 1051, (1,1,3): 1070, (1,2,2): 253}
observed_counts = {tuple(row['polarization_type']): row['candidate_count'] for row in summaries}
assert observed_counts == expected_counts
assert len(rows) == 4165
assert all(row['pairing_verified'] for row in rows)
assert all(row['mean_ell_squared_enhanced'] > row['mean_ell_squared_minimal_image'] for row in summaries)
assert all(row['upper_quartile_extra_symmetry_fraction'] > row['extra_passive_symmetry_fraction'] for row in summaries)
assert all(row['distance_log_image_correlation'] > 0 for row in summaries)
print('Phase 5 population checks passed.')

Phase 5 population checks passed.
